# 02c — Benefit Scoring & Scenario Preparation (Heuristic)

This companion notebook extends the Phase 2 scenario layer *without* modifying
the original `02_scenario_prep_and_risk_features.ipynb` pipeline.

Goals:

1. Load the canonical scenario table  
   `data/interim/trials_scenarios.parquet` (one row per `nct_id`).
2. Attach a simple, interpretable `benefit_score` based on trial phase and
   recruitment status.
3. Re-write the scenario parquet artifacts so future notebooks can treat
   `benefit_score` as a standard feature, alongside cost and enrollment
   feasibility.
4. (Optionally) act as a starting point for defining Scenario A/B slices in
   later cells.


In [1]:
# ============================================================
# Cell 1 — Load trials_scenarios from Phase 2
# ============================================================

from pathlib import Path
import pandas as pd

def log(msg: str) -> None:
    print(msg)

SCENARIOS_PARQUET = Path("data/interim/trials_scenarios.parquet")

if SCENARIOS_PARQUET.exists():
    trials_scenarios = pd.read_parquet(SCENARIOS_PARQUET)
    log(f"[Cell 1] Loaded trials_scenarios with shape {trials_scenarios.shape}")
else:
    raise FileNotFoundError(
        f"[Cell 1] Missing {SCENARIOS_PARQUET}. "
        "Run 02_scenario_prep_and_risk_features.ipynb first."
    )

trials_scenarios.head()

[Cell 1] Loaded trials_scenarios with shape (557292, 12)


,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score
0,NCT00000102,Congenital Adrenal Hyperplasia: Calcium Channe...,Completed,Phase 1/Phase 2,[Congenital Adrenal Hyperplasia],[Nifedipine],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.5,1.0
1,NCT00000104,Does Lead Burden Alter Neuropsychological Deve...,Completed,None,[Lead Poisoning],[ERP measures of attention and memory],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0
2,NCT00000105,Vaccination With Tetanus and KLH to Assess Imm...,Terminated,None,[Cancer],"[Intracel KLH Vaccine, Biosyn KLH, Montanide I...",[United States],"Masonic Cancer Center, University of Minnesota","Masonic Cancer Center, University of Minnesota",Global / Multi-Region,1.0,1.0
3,NCT00000106,41.8 Degree Centigrade Whole Body Hyperthermia...,Unknown status,N/A,[Rheumatic Diseases],[Whole body hyperthermia unit],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0
4,NCT00000107,Body Water Content in Cyanotic Congenital Hear...,Completed,None,"[Heart Defects, Congenital]",[],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0


### What Cell 1 Just Did

This step loaded the canonical Phase 2 scenario table:

- Read `data/interim/trials_scenarios.parquet` into `trials_scenarios`.
- Confirmed its shape and previewed the first few rows.

At this point we have one row per `nct_id`, with core fields such as:

- Trial context: `nct_id`, `brief_title`, `overall_status`, `phase`
- Clinical framing: `conditions`, `interventions`, `location_countries`
- Sponsor / region: `lead_sponsor`, `lead_sponsor_norm`, `region_label`
- Operations features: `estimated_trial_cost`, `enrollment_feasibility_score`

This notebook will build on top of this table without changing the original
Phase 2 notebook code.


In [2]:
# ============================================================
# Cell 2 — Add heuristic benefit_score and update scenarios
# ============================================================
#
# Purpose:
#   - Attach a simple, interpretable "benefit_score" to each trial,
#     based on its phase and recruitment status.
#   - Rewrite trials_scenarios.parquet and trials_scenarios_sample.parquet
#     to include this new feature.
#
# Heuristic:
#   - Later-phase trials typically have more mature evidence → higher weight.
#   - Actively recruiting or enrolling trials → higher near-term impact.
#   - Suspended/terminated/withdrawn trials → low benefit.
#
# NOTE:
#   This is an illustrative scoring function for experimentation,
#   not a clinical or commercial recommendation.

import numpy as np

def phase_weight(phase: str | float | None) -> float:
    """Map phase string to a coarse benefit weight in [0, 1]."""
    if phase is None or (isinstance(phase, float) and np.isnan(phase)):
        return 0.3

    p = str(phase).strip()

    mapping = {
        "Early Phase 1": 0.3,
        "Phase 1": 0.35,
        "Phase 1/Phase 2": 0.45,
        "Phase 2": 0.6,
        "Phase 2/Phase 3": 0.7,
        "Phase 3": 0.8,
        "Phase 4": 0.7,
        "N/A": 0.4,
        "None": 0.3,
    }

    return mapping.get(p, 0.4)  # fallback for odd / unexpected strings


def status_weight(status: str | float | None) -> float:
    """Map overall_status string to a coarse benefit weight in [0, 1]."""
    if status is None or (isinstance(status, float) and np.isnan(status)):
        return 0.3

    s = str(status).strip()

    high = {
        "Recruiting",
        "Enrolling by invitation",
        "Active, not recruiting",
    }
    medium = {
        "Not yet recruiting",
        "Completed",
        "Available",
        "Approved for marketing",
        "No longer available",
    }
    low = {
        "Unknown status",
        "Suspended",
        "Terminated",
        "Withdrawn",
        "Withheld",
        "Temporarily not available",
    }

    if s in high:
        return 1.0
    if s in medium:
        return 0.6
    if s in low:
        return 0.2

    # Fallback for rare or unexpected labels
    return 0.4


# --- Compute benefit_score on trials_scenarios -----------------------------

missing_cols = [c for c in ["phase", "overall_status"] if c not in trials_scenarios.columns]
if missing_cols:
    log(f"[Cell 2] WARNING: Missing columns for benefit_score: {missing_cols}")
else:
    log("[Cell 2] Computing benefit_score from phase and overall_status...")

    phase_w = trials_scenarios["phase"].apply(phase_weight)
    status_w = trials_scenarios["overall_status"].apply(status_weight)

    # Simple multiplicative combination, then clipped to [0, 1]
    benefit_raw = phase_w * status_w
    benefit_score = np.clip(benefit_raw, 0.0, 1.0)

    trials_scenarios["benefit_score"] = benefit_score

    log("[Cell 2] benefit_score added to trials_scenarios.")
    display(
        trials_scenarios[
            ["nct_id", "phase", "overall_status", "benefit_score"]
        ].head(10)
    )

    # --- Rewrite scenario artifacts to include benefit_score --------------

    SCENARIOS_PARQUET = Path("data/interim/trials_scenarios.parquet")
    SCENARIOS_SAMPLE_PARQUET = Path("data/interim/trials_scenarios_sample.parquet")

    # Persist full table
    SCENARIOS_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    trials_scenarios.to_parquet(SCENARIOS_PARQUET, index=False)
    log(
        f"[Cell 2] Re-wrote trials_scenarios to {SCENARIOS_PARQUET} "
        f"with shape {trials_scenarios.shape}"
    )

    # Rebuild sample
    sample_n = min(10000, len(trials_scenarios))
    trials_scenarios_sample = trials_scenarios.sample(n=sample_n, random_state=42)
    trials_scenarios_sample.to_parquet(SCENARIOS_SAMPLE_PARQUET, index=False)
    log(
        f"[Cell 2] Re-wrote trials_scenarios_sample ({sample_n} rows) "
        f"to {SCENARIOS_SAMPLE_PARQUET}"
    )

[Cell 2] Computing benefit_score from phase and overall_status...
[Cell 2] benefit_score added to trials_scenarios.


,nct_id,phase,overall_status,benefit_score
0,NCT00000102,Phase 1/Phase 2,Completed,0.27
1,NCT00000104,None,Completed,0.18
2,NCT00000105,None,Terminated,0.06
3,NCT00000106,N/A,Unknown status,0.08
4,NCT00000107,None,Completed,0.18
5,NCT00000108,N/A,Completed,0.24
6,NCT00000110,N/A,Completed,0.24
7,NCT00000111,Phase 1,Unknown status,0.07
8,NCT00000112,None,Unknown status,0.06
9,NCT00000113,Phase 3,Completed,0.48


[Cell 2] Re-wrote trials_scenarios to data/interim/trials_scenarios.parquet with shape (557292, 13)
[Cell 2] Re-wrote trials_scenarios_sample (10000 rows) to data/interim/trials_scenarios_sample.parquet


### What Cell 2 Just Did

This step attached a heuristic `benefit_score` to each trial in
`trials_scenarios` based on:

- **Phase** — later-phase trials (e.g., Phase 2/3) receive higher phase weights
  than early or unspecified phases.
- **Overall recruitment status** — trials that are recruiting, enrolling by
  invitation, or active receive higher status weights than completed,
  not-yet-recruiting, or discontinued studies.

The notebook multiplies these two weights and clips the result into \[0, 1\],
yielding a simple `benefit_score` that is easy to inspect and filter on in
downstream analysis.

Finally, it re-saved:

- `data/interim/trials_scenarios.parquet`  
- `data/interim/trials_scenarios_sample.parquet`

so future notebooks can treat `benefit_score` as another scenario feature
alongside cost and enrollment feasibility, without changing the original
Phase 2 pipeline code.


## Notebook Summary — Heuristic Benefit Scoring

This helper notebook extends the Phase 2 scenario layer in a non-invasive way.

Starting from the canonical scenario table
`data/interim/trials_scenarios.parquet`, it:

1. Loaded one-row-per-trial scenarios into `trials_scenarios`.
2. Computed a simple, interpretable `benefit_score` for each trial based on:
   - Clinical trial phase, and
   - Overall recruitment status.
3. Rewrote both the full scenario table and its 10k-row sample so that
   `benefit_score` is now a standard feature available to downstream notebooks.

The core Phase 2 notebook remains unchanged, while this companion notebook
provides an explicit, documented place for heuristic benefit logic that can be
refined or replaced over time.
